# MLP Predictions Dataset Generator

This notebook:
1. Loads the trained MLP model from models/Aug29/mlp
2. Loads data from data/Aug29/2_DyF_L2.jsonl
3. Generates predictions using the MLP model
4. Creates a dataset with: timestamp, FF_EE, FRaw, and predicted forces
5. Saves the dataset as a JSON file in the NoiceCalcExp folder

**Model Input Features:**
- POS[3:6]: Orientation angles (rx, ry, rz)
- X_DoubleDot_Cmds[0:6]: 6DOF acceleration commands

**Model Output:**
- FF_EE_Dynamic[0:6]: 6DOF predicted dynamic forces


In [ ]:
import json
import numpy as np
import joblib
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set up paths
data_path = Path('../data/Aug29/2_DyF_L2.jsonl')
model_path = Path('../models/Aug29/mlp/mlp_model.pkl')
scaler_path = Path('../models/Aug29/mlp/mlp_scaler.pkl')
output_path = Path('mlp_predictions_dataset.json')

print("Paths:")
print(f"  Data: {data_path}")
print(f"  Model: {model_path}")
print(f"  Scaler: {scaler_path}")
print(f"  Output: {output_path}")


Paths:
  Data: ..\data\Aug29\2_DyF_L2.jsonl
  MLP Model: ..\models\Aug29\mlp\mlp_model.pkl
  MLP Scaler: ..\models\Aug29\mlp\mlp_scaler.pkl
  Quasi-Static Model: ..\models\Aug29\1_Quasi_Static_Poly.pkl
  Output: mlp_predictions_dataset.jsonl


In [ ]:
# Load the MLP model and scaler
print("Loading MLP model and scaler...")
model = joblib.load(model_path)
scaler = joblib.load(scaler_path)
print("✓ Model and scaler loaded successfully")
print(f"Model type: {type(model)}")


Loading MLP model and scaler...
✓ MLP model and scaler loaded successfully
MLP Model type: <class 'sklearn.neural_network._multilayer_perceptron.MLPRegressor'>

Loading quasi-static model...
✓ Quasi-static model loaded successfully
Quasi-static Model type: <class 'sklearn.pipeline.Pipeline'>


In [3]:
# Load the data
print(f"\nLoading data from {data_path}...")
data_points = []
with open(data_path, 'r') as f:
    for line in f:
        if line.strip():
            data_points.append(json.loads(line))

print(f"✓ Loaded {len(data_points)} data points")
print(f"Sample keys: {list(data_points[0].keys())}")



Loading data from ..\data\Aug29\2_DyF_L2.jsonl...
✓ Loaded 48745 data points
Sample keys: ['timestamp', 'delta_time', 'POS', 'FRaw', 'FF_EE', 'FF_EE_Quasi', 'FF_EE_Dynamic', 'FF_GB', 'Foot_Pedal', 'X_Dot_Cmds', 'X_DoubleDot_Cmds']


In [4]:
# Extract features for MLP model
# Features: POS[3:6] (rx, ry, rz) + X_DoubleDot_Cmds[0:6] (6DOF accelerations)
print("\nExtracting features...")
features = []
for dp in data_points:
    # Extract orientation angles (rx, ry, rz) from POS[3:6]
    orientation = dp['POS'][3:6]
    # Extract acceleration commands
    accelerations = dp['X_DoubleDot_Cmds']
    # Combine: 3 orientation + 6 accelerations = 9 features
    feature_vector = orientation + accelerations
    features.append(feature_vector)

X = np.array(features)
print(f"✓ Feature matrix shape: {X.shape}")
print(f"  Expected: (N, 9) where N={len(data_points)}")



Extracting features...
✓ Feature matrix shape: (48745, 9)
  Expected: (N, 9) where N=48745


In [ ]:
# Scale features using the saved scaler
print("\nScaling features...")
X_scaled = scaler.transform(X)
print(f"✓ Scaled feature matrix shape: {X_scaled.shape}")



Scaling features for MLP model...
✓ Scaled feature matrix shape: (48745, 9)


In [ ]:
# Generate predictions
print("\nGenerating predictions...")
predictions = model.predict(X_scaled)
print(f"✓ Predictions shape: {predictions.shape}")
print(f"  Expected: (N, 6) where N={len(data_points)}")
print(f"  Predictions are FF_EE_Dynamic[0:6] (6DOF dynamic forces)")



Generating predictions...
  Predicting dynamic forces (MLP)...


  ✓ Dynamic predictions shape: (48745, 6)
  Predicting quasi-static forces (Polynomial)...
  ✓ Quasi-static predictions shape: (48745, 6)
  Combining quasi-static + dynamic...
  ✓ Final predictions shape: (48745, 6)

  Summary:
    - Quasi-static component: mean magnitude = 2.8370
    - Dynamic component: mean magnitude = 0.0381
    - Final predicted force: mean magnitude = 2.8479


In [ ]:
# Create dataset with timestamp, FF_EE, FRaw, and predicted forces
print("\nCreating dataset...")
dataset = []
for i, dp in enumerate(data_points):
    entry = {
        'timestamp': dp['timestamp'],
        'FF_EE': dp['FF_EE'],  # 6DOF filtered force/torque
        'FRaw': dp['FRaw'],    # 6DOF raw force/torque
        'predicted_force': predictions[i].tolist()  # 6DOF predicted dynamic force
    }
    dataset.append(entry)

print(f"✓ Created dataset with {len(dataset)} entries")
print(f"\nSample entry structure:")
print(json.dumps(dataset[0], indent=2))



Creating dataset...
✓ Created dataset with 48745 entries

Sample entry structure:
{
  "timestamp": 1756487196.7640388,
  "FF_EE": [
    0.9879126072567948,
    3.7641955158122897,
    12.112603442683385,
    -0.3596918246439192,
    -0.03493050580606379,
    0.0004874970312140867
  ],
  "FRaw": [
    0.9936548691101367,
    3.7755921681657556,
    12.127974707886766,
    -0.3608896542715033,
    -0.03288614704552899,
    0.0005862689838614064
  ],
  "predicted_force": [
    0.9757005121071974,
    3.7493237898161422,
    12.100692388447518,
    -0.3597884206050203,
    -0.03451420827012688,
    0.0002971980130493132
  ]
}


In [ ]:
# Save dataset to JSON file
print(f"\nSaving dataset to {output_path}...")
with open(output_path, 'w') as f:
    json.dump(dataset, f, indent=2)

print(f"✓ Dataset saved successfully!")
print(f"  File size: {output_path.stat().st_size / 1024 / 1024:.2f} MB")
print(f"  Total entries: {len(dataset)}")
print(f"\nDataset structure:")
print("  - timestamp: float")
print("  - FF_EE: [Fx, Fy, Fz, Mx, My, Mz] (6 values)")
print("  - FRaw: [Fx, Fy, Fz, Mx, My, Mz] (6 values)")
print("  - predicted_force: [Fx, Fy, Fz, Mx, My, Mz] (6 values)")



Saving dataset to mlp_predictions_dataset.jsonl...
✓ Dataset saved successfully as JSONL!
  File size: 21.04 MB
  Total entries: 48745
  Format: JSONL (one JSON object per line)

Dataset structure:
  - timestamp: float
  - FF_EE: [Fx, Fy, Fz, Mx, My, Mz] (6 values)
  - FRaw: [Fx, Fy, Fz, Mx, My, Mz] (6 values)
  - predicted_force: [Fx, Fy, Fz, Mx, My, Mz] (6 values)


In [9]:
# Quick statistics
print("\n=== Dataset Statistics ===")
print(f"Timestamps:")
print(f"  First: {dataset[0]['timestamp']:.6f}")
print(f"  Last: {dataset[-1]['timestamp']:.6f}")
print(f"  Duration: {(dataset[-1]['timestamp'] - dataset[0]['timestamp']):.2f} seconds")

print(f"\nFRaw ranges:")
for i, name in enumerate(['Fx', 'Fy', 'Fz', 'Mx', 'My', 'Mz']):
    values = [entry['FRaw'][i] for entry in dataset]
    print(f"  {name}: [{min(values):.4f}, {max(values):.4f}]")

print(f"\nFF_EE ranges:")
for i, name in enumerate(['Fx', 'Fy', 'Fz', 'Mx', 'My', 'Mz']):
    values = [entry['FF_EE'][i] for entry in dataset]
    print(f"  {name}: [{min(values):.4f}, {max(values):.4f}]")

print(f"\nPredicted force ranges:")
for i, name in enumerate(['Fx', 'Fy', 'Fz', 'Mx', 'My', 'Mz']):
    values = [entry['predicted_force'][i] for entry in dataset]
    print(f"  {name}: [{min(values):.4f}, {max(values):.4f}]")



=== Dataset Statistics ===
Timestamps:
  First: 1756487196.764039
  Last: 1756487391.739275
  Duration: 194.98 seconds

FRaw ranges:
  Fx: [-3.0957, 3.2356]
  Fy: [-1.2665, 7.1183]
  Fz: [10.0034, 14.1517]
  Mx: [-0.5668, -0.0251]
  My: [-0.2914, 0.1288]
  Mz: [-0.1339, 0.0838]

FF_EE ranges:
  Fx: [-2.0152, 1.9262]
  Fy: [1.2131, 5.2686]
  Fz: [11.4360, 12.4895]
  Mx: [-0.4263, -0.2252]
  My: [-0.1946, 0.0165]
  Mz: [-0.0846, 0.0299]

Predicted force ranges:
  Fx: [-1.9810, 1.8652]
  Fy: [1.2795, 5.2172]
  Fz: [11.4611, 12.4662]
  Mx: [-0.4238, -0.2305]
  My: [-0.1900, 0.0115]
  Mz: [-0.0830, 0.0267]
